# 00 — Prompts and Feedback

Inspect output rules, example code, task prompts, prompt strategies and execution feedback.


In [ ]:
import sys
from pathlib import Path
import numpy as np

# Ensure project root & src are in path
cwd = Path(".").resolve()
root_dir = cwd.parent if cwd.name == "notebooks" else cwd
src_dir = root_dir / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# Evict cached modules in case kernel was already running before disk update
for mod in [m for m in list(sys.modules.keys()) if m.startswith("evolution.infra.engines.llamea")]:
    del sys.modules[mod]

from evolution.domain.enums import NoiseEnvironment, PromptStrategy, SynthesisMode
from evolution.infra.engines.llamea.prompts import (
    build_task_prompt,
    build_example_prompt,
    build_format_prompt,
    FeedbackRenderer,
    META_FEEDBACK_DIVERSITY_INJECTION,
)
from shared.domain.noise import HeteroscedasticNoiseStrategy, NoNoiseStrategy
from shared.infra.problems.bbob import BBOBProblem
from IPython.display import display, Markdown

# Default sample parameters for inspection
problem_id = 1
dim = 2
lb = np.array([-5.0, -5.0])
ub = np.array([5.0, 5.0])
budget = 2000
sample_problem = BBOBProblem(problem_id=1, dim=2, noise_strategy=NoNoiseStrategy(), instance_id=1)

print("[OK] LLaMEA Prompt & Feedback builder loaded successfully.")


## 1. Output format


In [ ]:
format_prompt = build_format_prompt()
print(format_prompt)


## 2. Example optimizer


In [ ]:
example_prompt = build_example_prompt()
print(example_prompt)


## 3. Task prompt variants


In [ ]:
modes = [SynthesisMode.EXPLICIT, SynthesisMode.IMPLICIT]

for mode in modes:
    p = build_task_prompt(
        problem=sample_problem,
        mode=mode,
        strategy=PromptStrategy.BASELINE,
        budget_hint=budget,
    )
    print("=" * 80)
    print(f"=== SYNTHESIS MODE: {mode.value.upper()} (Baseline Strategy) ===")
    print("=" * 80)
    print(p)
    print()


## 4. Prompt strategies


In [ ]:
strategies = [
    PromptStrategy.BASELINE,
    PromptStrategy.VECTORIZATION,
    PromptStrategy.GUIDED,
    PromptStrategy.THINKING,
]

noisy_problem = BBOBProblem(problem_id=1, dim=2, noise_strategy=HeteroscedasticNoiseStrategy(0.1), instance_id=1)
for strat in strategies:
    p = build_task_prompt(
        problem=noisy_problem,
        mode=SynthesisMode.EXPLICIT,
        strategy=strat,
        budget_hint=budget,
    )
    print("=" * 80)
    print(f"=== NOISY MODE with STRATEGY: {strat.value.upper()} ===")
    print("=" * 80)
    print(p)
    print()


## 5. Assembled prompt


In [ ]:
def print_full_prompt(mode=SynthesisMode.EXPLICIT, noise_environment=NoiseEnvironment.NOISY, strategy=PromptStrategy.GUIDED, p_id=1, d=2, b=2000):
    noise_strat = HeteroscedasticNoiseStrategy(0.1) if noise_environment == NoiseEnvironment.NOISY else NoNoiseStrategy()
    prob = BBOBProblem(problem_id=p_id, dim=d, noise_strategy=noise_strat, instance_id=1)
    task = build_task_prompt(
        problem=prob,
        mode=mode,
        strategy=strategy,
        budget_hint=b,
    )
    fmt = build_format_prompt()
    ex = build_example_prompt()

    separator = "#" * 80
    output = f"""{separator}
# 1. TASK PROMPT (Problem Description & Strategy Guidance)
{separator}
{task}

{separator}
# 2. OUTPUT FORMAT RULES
{separator}
{fmt}

{separator}
# 3. CODE SKELETON EXAMPLE
{separator}
{ex}
"""
    print(output)
    return output

# Run for Noisy + Guided
prompt_text = print_full_prompt(mode=SynthesisMode.EXPLICIT, strategy=PromptStrategy.GUIDED)


## 6. Execution and stagnation feedback


In [ ]:
from shared.domain.noise import HeteroscedasticNoiseStrategy, NoNoiseStrategy
from shared.infra.problems.bbob import BBOBProblem
from evolution.infra.engines.llamea.prompts import (
    FeedbackRenderer,
    META_FEEDBACK_DIVERSITY_INJECTION,
)
from evolution.application.synthesis.candidate_evaluation import CandidateEvaluator

# Setup sample problem instances and feedback renderer
prob_clean = BBOBProblem(1, 2, NoNoiseStrategy(), 1)
prob_noisy = BBOBProblem(1, 2, HeteroscedasticNoiseStrategy(0.05), 1)
renderer = FeedbackRenderer()

dummy_code = """import numpy as np

class CovarianceOptimizer:
    def __init__(self):
        pass

    def __call__(self, problem, budget):
        lb = problem.lower_bound
        ub = problem.upper_bound
        dim = problem.dim
        pop = np.random.uniform(lb, ub, size=(50, dim))
        cov = np.cov(pop.T)
        inv_cov = np.linalg.inv(cov)
        best_x = pop[0]
        best_y = float(problem(best_x))
        return best_x, best_y"""

def display_feedback_examples():
    # 1. Clean Success
    print("=== 1. SUCCESSFUL RUN (Clean Objective) ===")
    print(renderer.render_success(final_error=0.0012, problem=prob_clean))
    print()

    # 2. Noisy Success (Large Error)
    print("=== 2. SUCCESSFUL RUN (Noisy Objective, Large Error) ===")
    print(renderer.render_success(final_error=0.8250, problem=prob_noisy))
    print()

    # 3. Failed Run (Runtime / Value Error with Code Context)
    print("=== 3. FAILED RUN (Runtime Exception with Context) ===")
    try:
        # Simulate execution error inside candidate code
        exec(dummy_code)
        raise ValueError("Matrix is singular and cannot be inverted")
    except Exception as e:
        tb_str = """Traceback (most recent call last):\n  File "<string>", line 14, in __call__\n    inv_cov = np.linalg.inv(cov)\nValueError: Matrix is singular and cannot be inverted"""
        code_context = CandidateEvaluator.extract_code_context(tb_str, dummy_code)
        print(
            renderer.render_failure(
                error_type="ValueError",
                error_message="Matrix is singular and cannot be inverted",
                problem=prob_noisy,
                code_context=code_context,
            )
        )
    print()

    # 4. Stagnation Meta-Feedback
    print("=== 4. STAGNATION OVERRIDE (Meta-Feedback) ===")
    print(META_FEEDBACK_DIVERSITY_INJECTION)

display_feedback_examples()
